# Docker od podstaw

**Kurs:** Podstawy konteneryzacji i narzędzi DevOps  
**Poziom:** początkujący (znajomość terminala i podstaw Pythona)  
**Czas:** ok. 100–130 minut + zadania własne  
**Forma:** teoria → przykład → wykonanie → pytania kontrolne

W trakcie zajęć użyjemy gotowych obrazów, uruchomimy serwer WWW, napiszemy **własny Dockerfile** i skonteneryzujemy prostą aplikację HTTP w Pythonie. Następnie przećwiczymy trwałość danych, sieci i Docker Compose.

> **Ważne:** polecenia `docker` wykonują się **w systemie, w którym działa kernel Jupytera**. Jeśli używasz Google Colab, zdalnego Jupytera lub Jupytera uruchomionego w kontenerze bez dostępu do Docker Engine, część ćwiczeń nie zadziała. Najprościej uruchomić notebook lokalnie po zainstalowaniu Docker Desktop / Docker Engine. Część dotycząca Dockera wymaga połączenia z działającym demonem.

## Cele

Po wykonaniu notebooka student powinien potrafić:

1. Wyjaśnić różnicę między **obrazem (image)**, **kontenerem (container)** i **rejestrem (registry)**.
2. Pobrać obraz, uruchomić kontener, sprawdzić jego logi i zakończyć działanie.
3. Wyjaśnić mapowanie portów oraz różnicę między `EXPOSE` a `-p`.
4. Napisać `Dockerfile`, zbudować obraz i uruchomić aplikację w kontenerze.
5. Rozróżnić **volume** i **bind mount** oraz wykorzystać podstawową sieć Dockera.
6. Uruchomić usługi deklaratywnie za pomocą **Docker Compose**.
7. Wskazać podstawowe praktyki bezpieczeństwa i debugowania.

### Przed rozpoczęciem

- Zainstaluj **Docker Desktop** (Windows/macOS) lub **Docker Engine z Compose** (Linux).
- Uruchom Docker Desktop / usługę Docker Engine.
- Przygotuj lokalny Jupyter Notebook / JupyterLab z kernelem Python 3.
- Zapewnij dostęp do internetu: ćwiczenia pobierają obrazy kontenerów.
- Porty lokalne `8764`, `8765`, `8766` powinny być wolne; jeśli są zajęte, zmień tylko port po lewej stronie mapowania.

**Zasada bezpieczeństwa:** laboratorium tworzy tylko zasoby z prefiksem `docker-edu-`. Nie wykonujemy globalnych komend typu `docker system prune -a`.

## 1. Co to jest Docker?

**Konteneryzacja** pozwala uruchamiać aplikacje wraz z ich zależnościami w izolowanych środowiskach. Dzięki temu łatwiej uzyskać podobne działanie na laptopie, serwerze i w środowisku CI.

| Pojęcie | Wyjaśnienie | Przykład |
|---|---|---|
| **Dockerfile** | Przepis opisujący sposób budowy obrazu | `FROM python:3.12-slim` |
| **Image** | Niemutowalny szablon systemu plików i konfiguracji | `nginx:alpine` |
| **Container** | Instancja obrazu z procesem/procesami | serwer `nginx` uruchomiony z obrazu |
| **Registry** | Repozytorium obrazów | Docker Hub |
| **Volume** | Zarządzane przez Dockera miejsce na trwałe dane | `docker-edu-data` |
| **Docker Compose** | Deklaratywne uruchamianie usług z YAML | `compose.yaml` |

```text
Dockerfile --docker build--> IMAGE --docker run--> CONTAINER
                             ^                      |
                             |                      +--> sieć, porty, logi
                        Docker Registry             +--> volume / bind mount
```

**Kontener a maszyna wirtualna:** kontenery linuksowe współdzielą jądro Linuksa, natomiast typowa maszyna wirtualna uruchamia własny system operacyjny i jądro. Na macOS i Windows Docker Desktop zwykle zapewnia środowisko linuksowe za pośrednictwem wirtualizacji.

**Pytanie:** Czy jeden obraz może posłużyć do uruchomienia kilku kontenerów? Tak — każdy z nich będzie osobną instancją.

## 2. Test środowiska i pomocnicza funkcja wykonująca polecenia

W notebooku używamy `subprocess`, aby uruchamiać dokładnie takie same polecenia Docker CLI jak w terminalu. Dzięki temu materiał działa w Pythonie i nie wymaga składni `%%bash` (która może być niedostępna w Windows).

Każdy przykład pokazuje odpowiednik polecenia terminalowego w komentarzu. Funkcja `docker_cmd()` wyświetli faktycznie uruchamiane polecenie i wynik działania.

In [1]:
from pathlib import Path
import json
import shlex
import shutil
import subprocess
import time
import urllib.request

# Notebook utworzy pliki aplikacji w tym podkatalogu.
PROJECT_DIR = Path.cwd() / "docker_basics_lab"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


def docker_cmd(*args, cwd=None, check=True):
    """Uruchamia docker CLI i pokazuje wynik; args to kolejne argumenty polecenia."""
    if shutil.which("docker") is None:
        raise RuntimeError("Nie znaleziono polecenia 'docker'. Zainstaluj Docker i uruchom Jupyter lokalnie.")

    command = ["docker", *map(str, args)]
    print("$", shlex.join(command))
    completed = subprocess.run(
        command, cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        check=False
    )
    print(completed.stdout.strip() or "(brak komunikatów)")
    if check and completed.returncode != 0:
        raise RuntimeError(f"Docker zakończył działanie z kodem {completed.returncode}.")
    return completed


def http_get_json(url, attempts=15):
    """Pobiera JSON z lokalnego serwera, z krótkim retry na czas startu kontenera."""
    for attempt in range(attempts):
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return json.loads(response.read().decode("utf-8"))
        except (OSError, ValueError):
            if attempt == attempts - 1:
                raise
            time.sleep(0.4)

print("Katalog laboratoryjny:", PROJECT_DIR.resolve())
print("Docker CLI:", shutil.which("docker") or "NIE ZNALEZIONO")

Katalog laboratoryjny: /app/docker_basics_lab
Docker CLI: NIE ZNALEZIONO


In [2]:
# Terminal: docker --version
# Terminal: docker info --format '{{.ServerVersion}}'
# Terminal: docker compose version
# Pierwsza komenda sprawdza klienta, druga połączenie z silnikiem (daemon),
# a trzecia dostępność wtyczki Docker Compose.
if shutil.which("docker"):
    docker_cmd("--version", check=False)
    docker_cmd("info", "--format", "{{.ServerVersion}}", check=False)
    docker_cmd("compose", "version", check=False)
else:
    print("Najpierw zainstaluj Docker Desktop lub Docker Engine.")

Najpierw zainstaluj Docker Desktop lub Docker Engine.


**Interpretacja:** wynik `docker --version` oznacza, że masz klienta CLI. Nie oznacza jeszcze, że działa Docker Engine. Jeśli `docker info` zwraca błąd połączenia, uruchom Docker Desktop lub usługę `dockerd` i dopiero kontynuuj.

## 3. Pierwszy kontener: `hello-world`

Zaczynamy od obrazu zawierającego program testowy. `docker run` automatycznie pobierze obraz, jeśli nie ma go lokalnie.

- `docker pull IMAGE` — pobiera obraz.
- `docker run IMAGE` — tworzy i uruchamia kontener.
- `--rm` — automatycznie usuwa kontener po zakończeniu; **nie usuwa obrazu**.

In [ ]:
# Terminal: docker pull hello-world
# Pobranie obrazu z rejestru.
docker_cmd("pull", "hello-world")

In [ ]:
# Terminal: docker run --rm hello-world
# Kontener uruchamia program, drukuje komunikat i kończy pracę.
docker_cmd("run", "--rm", "hello-world")

In [ ]:
# Terminal: docker image ls
# Terminal: docker ps
# Terminal: docker ps -a
# Obraz pozostaje; po --rm kontenera nie ma już na liście.
docker_cmd("image", "ls")
docker_cmd("ps")
docker_cmd("ps", "-a")

**Pomyśl:** dlaczego `docker ps` nie wyświetla ukończonych kontenerów? Domyślnie pokazuje wyłącznie kontenery aktualnie działające. `docker ps -a` pokazuje także zatrzymane.

## 4. Uruchomienie serwera WWW: Nginx

W tym ćwiczeniu użyjemy gotowego obrazu `nginx:alpine`. Serwer Nginx działa wewnątrz kontenera na porcie `80`. Do przeglądarki na naszym komputerze udostępnimy go pod `http://127.0.0.1:8764`.

```bash
docker run -d --name docker-edu-nginx -p 127.0.0.1:8764:80 nginx:alpine
```

- `-d` (`--detach`) — start w tle.
- `--name` — własna nazwa kontenera, używana w dalszych poleceniach.
- `-p HOST_IP:HOST_PORT:CONTAINER_PORT` — publikacja portu z kontenera na hoście.
- `127.0.0.1` ogranicza dostęp do lokalnego komputera. Przy `0.0.0.0` serwer może być dostępny także z sieci, zależnie od konfiguracji.

In [ ]:
# Terminal: docker run -d --name docker-edu-nginx -p 127.0.0.1:8764:80 nginx:alpine
# Pierwsze uruchomienie pobierze obraz Nginx, jeżeli nie ma go lokalnie.
docker_cmd("run", "-d", "--name", "docker-edu-nginx",
           "-p", "127.0.0.1:8764:80", "nginx:alpine")

In [ ]:
# Terminal: docker ps
# Terminal: docker logs docker-edu-nginx
# Terminal: docker port docker-edu-nginx
# Weryfikacja procesu, logów i mapowania portów.
docker_cmd("ps")
docker_cmd("logs", "docker-edu-nginx")
docker_cmd("port", "docker-edu-nginx")

In [ ]:
# Odpowiednik: curl http://127.0.0.1:8764/
# Biblioteka urllib jest dostępna w standardowej instalacji Pythona.
with urllib.request.urlopen("http://127.0.0.1:8764/", timeout=10) as response:
    html = response.read().decode("utf-8")
print("Kod HTTP:", response.status)
print("Fragment strony:", html[:350])

### Diagnostyka i polecenie `exec`

`docker exec` wykonuje nowy proces **wewnątrz już działającego kontenera**. Może to być powłoka `sh`, odczyt pliku lub inny program. W Jupyterze lepiej uruchamiać polecenia jednorazowe; interaktywną powłokę `docker exec -it ... sh` otwieraj w zwykłym terminalu.

In [ ]:
# Terminal: docker exec docker-edu-nginx sh -c 'id && ls -la /usr/share/nginx/html'
# Komenda 'id' pokazuje użytkownika; 'ls' listing plików witryny.
docker_cmd("exec", "docker-edu-nginx", "sh", "-c",
           "id && ls -la /usr/share/nginx/html")

# Terminal: docker inspect docker-edu-nginx --format '{{.State.Status}}'
docker_cmd("inspect", "docker-edu-nginx", "--format", "{{.State.Status}}")

### Zatrzymywanie i usuwanie kontenera

`docker stop` wysyła sygnał zakończenia i czeka na zatrzymanie. `docker rm` usuwa zatrzymany kontener. Obraz nadal pozostaje i można z niego uruchomić nowy kontener.

In [ ]:
# Terminal: docker stop docker-edu-nginx
# Terminal: docker rm docker-edu-nginx
# Cleanup dotyczy wyłącznie naszego kontenera laboratoryjnego.
docker_cmd("stop", "docker-edu-nginx", check=False)
docker_cmd("rm", "docker-edu-nginx", check=False)

## 5. Własna aplikacja Python: od kodu do obrazu

Napiszemy mini-API HTTP bez dodatkowych bibliotek. Ścieżka `/` wyświetla komunikat i nazwę hosta (kontenera), a `/health` odpowiada, czy proces serwera działa.

Użycie wyłącznie standardowej biblioteki Pythona pozwala skupić się na Dockerze zamiast na instalacji zależności. W systemie produkcyjnym zwykle wybiera się pełniejszy framework i serwer aplikacyjny.

In [ ]:
# Ten kod jest zapisywany do pliku app.py w katalogu projektu.
# Przeanalizuj komentarze w pliku przed dalszą częścią zajęć.
app_code = r'''"""Prosty serwer HTTP do ćwiczeń z Dockerem (tylko laboratorium)."""

from http.server import BaseHTTPRequestHandler, HTTPServer
import json
import os
import socket


class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        # Dwa proste endpointy: / oraz /health.
        if self.path not in ("/", "/health"):
            self.send_error(404, "Nieznana sciezka")
            return

        payload = (
            {"status": "ok"}
            if self.path == "/health"
            else {
                "message": os.environ.get("APP_MESSAGE", "Witaj z Dockera!"),
                "hostname": socket.gethostname(),
                "path": self.path,
            }
        )
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, fmt, *args):
        # Wypisujemy logi na stdout — przechwyci je 'docker logs'.
        print(fmt % args, flush=True)


if __name__ == "__main__":
    # 0.0.0.0 jest wymagane, aby serwer nasłuchiwał na interfejsie
    # sieciowym kontenera; 127.0.0.1 w kontenerze nie wystarczy.
    print("API nasluchuje na 0.0.0.0:8000", flush=True)
    HTTPServer(("0.0.0.0", 8000), Handler).serve_forever()
'''
(PROJECT_DIR / "app.py").write_text(app_code, encoding="utf-8")
print((PROJECT_DIR / "app.py").read_text(encoding="utf-8"))

### Plik `Dockerfile`

Dockerfile to instrukcje budowy obrazu, wykonywane w podanej kolejności:

| Instrukcja | Znaczenie |
|---|---|
| `FROM` | obraz bazowy, tutaj środowisko Pythona |
| `WORKDIR` | katalog roboczy wewnątrz obrazu |
| `COPY` | skopiowanie plików do obrazu |
| `RUN` | operacja wykonywana **podczas budowania obrazu** |
| `USER` | użytkownik używany dla późniejszych instrukcji i przy uruchomieniu |
| `ENV` | zmienna środowiskowa w obrazie |
| `EXPOSE` | informacja o porcie aplikacji; **nie** publikuje portu na hoście |
| `CMD` | domyślne polecenie uruchamiane **przy starcie kontenera** |

Celowo tworzymy użytkownika nieuprzywilejowanego. Uruchamianie całej aplikacji jako `root` zwykle nie jest potrzebne.

In [ ]:
# Tworzymy plik Dockerfile (bez rozszerzenia).
dockerfile_content = r'''# Obraz bazowy: oficjalny Python, wariant slim.
FROM python:3.12-slim

# Wszystkie kolejne instrukcje dotyczą katalogu /app.
WORKDIR /app

# Do obrazu kopiujemy tylko kod wymagany do uruchomienia API.
COPY app.py .

# RUN działa w trakcie budowania obrazu — tworzymy zwykłego użytkownika.
RUN useradd --create-home --uid 10001 appuser

# Serwer uruchomi się z mniejszymi uprawnieniami niż root.
USER appuser
ENV PYTHONUNBUFFERED=1

# EXPOSE nie zastępuje -p / ports w Docker Compose.
EXPOSE 8000

# CMD jest wykonywany, gdy kontener startuje.
CMD ["python", "app.py"]
'''
(PROJECT_DIR / "Dockerfile").write_text(dockerfile_content, encoding="utf-8")
print((PROJECT_DIR / "Dockerfile").read_text(encoding="utf-8"))

### Po co `.dockerignore`?

Docker buduje obraz na podstawie tzw. **build context** — katalogu przekazanego do `docker build`. Plik `.dockerignore` określa elementy, których nie chcemy przekazywać w kontekście (np. lokalna `.venv`, katalog `.git`, pliki `.env` z sekretami). Nie jest on odpowiednikiem izolacji uprawnień: trzeba nadal uważać na to, co aplikacja kopiuje do obrazu.

In [ ]:
ignore_content = r'''# Nie wysyłamy środowiska wirtualnego i plików tymczasowych do build context.
.venv/
__pycache__/
*.pyc
.git/
.env
*.log
*.ipynb
'''
(PROJECT_DIR / ".dockerignore").write_text(ignore_content, encoding="utf-8")
print((PROJECT_DIR / ".dockerignore").read_text(encoding="utf-8"))

### Budowa obrazu: `docker build`

```bash
cd docker_basics_lab
docker build -t docker-edu-api:1.0 .
```

- `-t` nadaje nazwę i tag obrazowi.
- `.` to **kontekst budowania**: bieżący katalog projektu.
- Zmiana kodu wymaga ponownego zbudowania obrazu, jeśli korzystasz z `COPY`.
- Docker wykorzystuje pamięć podręczną (cache) warstw, aby ograniczyć ponowną pracę.

In [ ]:
# Terminal (z katalogu projektu): docker build -t docker-edu-api:1.0 .
# Przy pierwszym uruchomieniu pobierany jest obraz bazowy Pythona.
docker_cmd("build", "-t", "docker-edu-api:1.0", ".", cwd=PROJECT_DIR)

In [ ]:
# Terminal: docker image ls docker-edu-api
# Terminal: docker history docker-edu-api:1.0
# History pomaga zobaczyć instrukcje składające się na obraz.
docker_cmd("image", "ls", "docker-edu-api")
docker_cmd("history", "docker-edu-api:1.0")

## 6. Uruchomienie własnego API w kontenerze

Udostępnimy port kontenera `8000` na porcie hosta `8765`. Zmienną `APP_MESSAGE` przekażemy do procesu wewnątrz kontenera za pomocą `-e`.

```bash
docker run -d --name docker-edu-api -p 127.0.0.1:8765:8000 \
  -e APP_MESSAGE='Witaj z kontenera!' docker-edu-api:1.0
```

**Uwaga:** wartości `-e` nadają się do ćwiczeń, ale nie należy przechowywać w ten sposób sekretów w logach, historii poleceń ani w obrazie. W środowisku produkcyjnym używaj odpowiednich mechanizmów zarządzania sekretami.

In [ ]:
# Terminal: docker run -d --name docker-edu-api -p 127.0.0.1:8765:8000 \
#             -e APP_MESSAGE='Witaj z kontenera!' docker-edu-api:1.0
# Oczekiwany wynik: identyfikator uruchomionego kontenera.
docker_cmd("run", "-d", "--name", "docker-edu-api",
           "-p", "127.0.0.1:8765:8000",
           "-e", "APP_MESSAGE=Witaj z kontenera!",
           "docker-edu-api:1.0")

In [ ]:
# Terminal: curl http://127.0.0.1:8765/
# Oczekujemy JSON z komunikatem oraz nazwą hosta kontenera.
result = http_get_json("http://127.0.0.1:8765/")
print(json.dumps(result, ensure_ascii=False, indent=2))
assert result["message"] == "Witaj z kontenera!"

# Terminal: curl http://127.0.0.1:8765/health
health = http_get_json("http://127.0.0.1:8765/health")
print("Health check:", health)
assert health == {"status": "ok"}

In [ ]:
# Terminal: docker logs docker-edu-api
# Terminal: docker exec docker-edu-api id
# Terminal: docker inspect docker-edu-api --format '{{.State.Status}}'
# Odczytujemy logi HTTP i weryfikujemy, że proces nie jest uruchomiony jako root.
docker_cmd("logs", "docker-edu-api")
docker_cmd("exec", "docker-edu-api", "id")
docker_cmd("inspect", "docker-edu-api", "--format", "{{.State.Status}}")

### Ważna obserwacja: obraz a kontener

Zmiana pliku `app.py` **na hoście** nie zmienia automatycznie wcześniej zbudowanego obrazu. Z kolei zmiany w warstwie zapisu konkretnego kontenera zwykle znikają po jego usunięciu (chyba że zapisano je na wolumenie lub w innym trwałym miejscu).

**Eksperyment:** zmień domyślny komunikat w `app.py`, zbuduj obraz o nowym tagu `docker-edu-api:1.1`, a następnie uruchom go pod inną nazwą i portem. Pamiętaj, że `APP_MESSAGE` przekazana przez `-e` przesłania wartość domyślną z kodu.

In [ ]:
# Terminal: docker stop docker-edu-api
# Terminal: docker rm docker-edu-api
# Zatrzymujemy i usuwamy tylko nasz kontener z API.
docker_cmd("stop", "docker-edu-api", check=False)
docker_cmd("rm", "docker-edu-api", check=False)

## 7. Trwałość danych: volume i bind mount

Kontener jest z założenia łatwy do odtworzenia. Nie należy przechowywać ważnych danych wyłącznie w jego zapisywalnej warstwie.

**Volume** to lokalizacja danych zarządzana przez Docker Engine. **Bind mount** podłącza do kontenera konkretny katalog/pliki z systemu hosta. Dla przykładu:

```text
Kontener 1 ----┐
               ├---- named volume "docker-edu-data" ---- plik notatka.txt
Kontener 2 ----┘
```

**Ćwiczenie:** utwórz volume, zapisz plik w jednym krótkotrwałym kontenerze i odczytaj go w drugim. Każdy kontener zakończy się i zostanie usunięty dzięki `--rm`, ale volume pozostanie.

In [ ]:
# Terminal: docker volume create docker-edu-data
# Terminal: docker run --rm --mount type=volume,source=docker-edu-data,target=/data \
#             alpine:3.21 sh -c 'echo Trwale_dane > /data/notatka.txt'
docker_cmd("volume", "create", "docker-edu-data")
docker_cmd("run", "--rm", "--mount", "type=volume,source=docker-edu-data,target=/data",
           "alpine:3.21", "sh", "-c", "echo Trwale_dane > /data/notatka.txt")

In [ ]:
# Terminal: docker run --rm --mount type=volume,source=docker-edu-data,target=/data \
#             alpine:3.21 cat /data/notatka.txt
# Nowy kontener, lecz dane zapisane wcześniej nadal istnieją.
docker_cmd("run", "--rm", "--mount", "type=volume,source=docker-edu-data,target=/data",
           "alpine:3.21", "cat", "/data/notatka.txt")
docker_cmd("volume", "inspect", "docker-edu-data")

### Bind mount — katalog z komputera hosta

Poniższy przykład montuje katalog projektu tylko do odczytu (`readonly`). Kod dynamicznie wyznacza ścieżkę lokalną, więc nie musisz wpisywać jej ręcznie. W Docker Desktop katalog hosta musi być udostępniony do montowania.

In [ ]:
# Terminal (Linux/macOS, z katalogu notebooka):
# docker run --rm --mount "type=bind,source=$(pwd)/docker_basics_lab,target=/lab,readonly" alpine:3.21 ls -la /lab
# W notebooku używamy Path.resolve() — działa także dla ścieżek Windows.
mount_spec = f"type=bind,source={PROJECT_DIR.resolve()},target=/lab,readonly"
docker_cmd("run", "--rm", "--mount", mount_spec, "alpine:3.21", "ls", "-la", "/lab")

**Porównaj:**

| Cecha | Named volume | Bind mount |
|---|---|---|
| Kto wskazuje lokalizację danych? | Docker | Użytkownik/ścieżka hosta |
| Typowe zastosowanie | Bazy danych, trwałe pliki aplikacji | Praca z kodem podczas developmentu |
| Los po usunięciu kontenera | Zwykle pozostaje | Katalog hosta pozostaje |

**Uwaga:** następna komenda usuwa **tylko** ćwiczebny volume, razem z zapisanymi w nim danymi.

In [ ]:
# Terminal: docker volume rm docker-edu-data
# Celowo usuwamy dane z tego laboratorium. W prawdziwym projekcie
# należy najpierw wykonać kopię zapasową.
docker_cmd("volume", "rm", "docker-edu-data", check=False)

## 8. Sieci i komunikacja między kontenerami

Kontenery podłączone do **tej samej sieci użytkownika** (user-defined bridge) mogą zwykle rozwiązywać wzajemnie swoje nazwy przez wewnętrzny DNS Dockera. Nie musimy publikować portu aplikacji na hoście, aby inny kontener w tej samej sieci się z nią połączył.

W eksperymencie uruchomimy Nginx wewnątrz sieci `docker-edu-net`, a potem pobierzemy jego stronę z krótkotrwałego kontenera BusyBox.

In [ ]:
# Terminal: docker network create docker-edu-net
# Terminal: docker run -d --name docker-edu-service --network docker-edu-net nginx:alpine
# Nginx nie ma tu -p, więc nie publikujemy portu na hoście.
docker_cmd("network", "create", "docker-edu-net")
docker_cmd("run", "-d", "--name", "docker-edu-service",
           "--network", "docker-edu-net", "nginx:alpine")

In [ ]:
# Terminal: docker run --rm --network docker-edu-net busybox:1.36 \
#             wget -qO- http://docker-edu-service/
# BusyBox korzysta z nazwy kontenera jako adresu DNS.
docker_cmd("run", "--rm", "--network", "docker-edu-net",
           "busybox:1.36", "wget", "-qO-", "http://docker-edu-service/")

In [ ]:
# Terminal: docker stop docker-edu-service
# Terminal: docker rm docker-edu-service
# Terminal: docker network rm docker-edu-net
# Usuwamy tylko sieć i kontener utworzone w naszym laboratorium.
docker_cmd("stop", "docker-edu-service", check=False)
docker_cmd("rm", "docker-edu-service", check=False)
docker_cmd("network", "rm", "docker-edu-net", check=False)

## 9. Docker Compose: konfiguracja zamiast wielu poleceń

Docker Compose pozwala opisać usługi i ich konfigurację w pliku `compose.yaml`. Definicja może obejmować obrazy, budowanie, sieci, porty, zmienne środowiskowe i wolumeny.

W tym laboratorium definiujemy dwie usługi:

- `api` — nasza aplikacja Python, budowana lokalnie.
- `client` — jednorazowe narzędzie do odpytania API z wewnętrznej sieci Compose. Ma profil `tools`, więc nie startuje podczas zwykłego `up`.

Wewnątrz sieci Compose nazwa usługi `api` jest adresem DNS, dlatego klient używa `http://api:8000/health` zamiast `localhost`.

In [ ]:
# Utworzenie pliku compose.yaml.
compose_content = r'''services:
  api:
    build: .
    ports:
      # Serwis dostępny tylko na localhost hosta.
      - "127.0.0.1:8766:8000"
    environment:
      APP_MESSAGE: "Uruchomiono przez Docker Compose"

  client:
    image: busybox:1.36
    profiles: ["tools"]
    depends_on:
      - api
    # W kontenerze 'client' nazwę 'api' rozwiązuje wewnętrzny DNS.
    command: ["wget", "-qO-", "http://api:8000/health"]
'''
(PROJECT_DIR / "compose.yaml").write_text(compose_content, encoding="utf-8")
print((PROJECT_DIR / "compose.yaml").read_text(encoding="utf-8"))

In [ ]:
# Terminal (w katalogu projektu): docker compose -p docker-edu config
# Walidacja YAML i rozwiązanej konfiguracji projektu.
docker_cmd("compose", "-p", "docker-edu", "config", cwd=PROJECT_DIR)

In [ ]:
# Terminal (w katalogu projektu): docker compose -p docker-edu up -d --build api
# --build wymusza weryfikację / przebudowanie obrazu aplikacji.
docker_cmd("compose", "-p", "docker-edu", "up", "-d", "--build", "api", cwd=PROJECT_DIR)
docker_cmd("compose", "-p", "docker-edu", "ps", cwd=PROJECT_DIR)

In [ ]:
# Terminal: curl http://127.0.0.1:8766/
# Test usługi opublikowanej przez Compose.
api_data = http_get_json("http://127.0.0.1:8766/")
print(json.dumps(api_data, indent=2, ensure_ascii=False))
assert api_data["message"] == "Uruchomiono przez Docker Compose"

In [ ]:
# Terminal (w katalogu projektu): docker compose -p docker-edu run --rm client
# Klient łączy się z API po nazwie usługi, nie przez port hosta.
docker_cmd("compose", "-p", "docker-edu", "run", "--rm", "client", cwd=PROJECT_DIR)

# Terminal (w katalogu projektu): docker compose -p docker-edu logs --tail=20 api
docker_cmd("compose", "-p", "docker-edu", "logs", "--tail=20", "api", cwd=PROJECT_DIR)

### `docker compose down` a dane trwałe

`docker compose down` zatrzymuje i usuwa kontenery oraz sieci stworzone przez projekt Compose. Nie usuwa automatycznie nazwanych wolumenów. Opcja `down -v` usuwa również wolumeny zarządzane przez projekt — **może trwale usunąć dane**. W naszym przykładzie nie ma wolumenu, więc wystarczy `down`.

In [ ]:
# Terminal (w katalogu projektu): docker compose -p docker-edu down
# Zatrzymanie i usunięcie zasobów Compose utworzonych w tej części zajęć.
docker_cmd("compose", "-p", "docker-edu", "down", cwd=PROJECT_DIR, check=False)

## 10. Podstawowe dobre praktyki

1. **Małe, oficjalne obrazy bazowe:** wybieraj adekwatną wersję i aktualizuj ją; obrazy `slim` lub `alpine` mogą zmniejszać rozmiar, ale mają różne ograniczenia.
2. **Nie uruchamiaj procesu jako root bez potrzeby:** `USER` w naszym Dockerfile redukuje uprawnienia procesu.
3. **Nie zapisuj haseł w obrazie i nie commituj `.env`:** stosuj przeznaczone do tego mechanizmy sekretów.
4. **Używaj `.dockerignore`:** ograniczaj build context i przypadkowe kopiowanie zbędnych danych.
5. **Pamiętaj o trwałości danych:** ważne pliki powinny znajdować się na wolumenach / zewnętrznych usługach, a nie tylko w warstwie zapisu kontenera.
6. **Korzystaj z logów:** aplikacja powinna wypisywać użyteczne komunikaty na stdout/stderr.
7. **Unikaj zbędnie otwartych portów:** na potrzeby lokalnych zajęć używaliśmy `127.0.0.1`.
8. **Nie traktuj Dockera jak pełnego mechanizmu bezpieczeństwa:** dostęp do demona Docker daje szerokie uprawnienia; nie uruchamiaj nieznanych obrazów bez weryfikacji.
9. **Dla powtarzalności pinuj wersje lub digesty:** `latest` może wskazywać na inny obraz w przyszłości.

### Typowe problemy i diagnoza

| Objaw | Możliwa przyczyna | Co sprawdzić? |
|---|---|---|
| `Cannot connect to the Docker daemon` | silnik Docker nie działa lub brak uprawnień | Docker Desktop / `docker info` |
| `port is already allocated` | port hosta jest zajęty | zmień np. `8765` na `8875` |
| `No such container` | kontener usunięto lub ma inną nazwę | `docker ps -a` |
| `Connection refused` | aplikacja nie działa lub nie nasłuchuje właściwie | `docker logs`, `docker ps`, adres `0.0.0.0` w aplikacji |
| `pull access denied` | zła nazwa obrazu lub brak autoryzacji | popraw nazwę, sprawdź uprawnienia do registry |
| Kod się zmienił, aplikacja nadal stara | obraz nie został przebudowany | `docker build` lub `docker compose up --build` |

## 11. Ćwiczenia

Ćwiczenia są przeznaczone do samodzielnej pracy. Oddaj pliki, komendy i krótkie odpowiedzi na pytania. Po każdym zadaniu usuń tylko utworzone przez siebie testowe kontenery.

### Poziom 1 — podstawowe polecenia

Uruchom kontener z obrazu `alpine:3.21`, który wypisze zawartość `/etc/os-release` i następnie się usunie. Podaj różnicę między `docker ps` a `docker ps -a`.

**Wskazówka:** `docker run --rm ... cat /etc/os-release`.

### Poziom 2 — konfiguracja i porty

Uruchom dwa kontenery `nginx:alpine`, każdy pod inną nazwą i portem hosta, np. `8871` oraz `8872`. Sprawdź oba adresy w przeglądarce. Wyjaśnij, dlaczego oba kontenery mogą korzystać z portu `80` wewnątrz kontenera.

### Poziom 3 — własny obraz

Rozbuduj `app.py` o endpoint `/info`, który zwróci JSON z polami `version`, `author` i `environment`. Zbuduj obraz `docker-edu-api:2.0`, uruchom go i pokaż działanie endpointu. **Nie kopiuj do obrazu plików `.env`.**

### Poziom 4 — trwałe dane

Utwórz volume `docker-edu-exercise-data`. Uruchom dwa niezależne kontenery, tak aby pierwszy zapisał w wolumenie plik `results.txt`, a drugi go odczytał. Pokaż, że usunięcie pierwszego kontenera nie usuwa zapisanych danych.

### Poziom 5 — Compose i sieć

Rozbuduj `compose.yaml` o trzecią usługę (np. `nginx`), udostępnij ją wyłącznie na `127.0.0.1:8873` i sprawdź komunikację pomiędzy usługami z użyciem ich nazw DNS. Dodaj do sprawozdania wynik `docker compose ps`, wybrane logi i krótkie uzasadnienie, kiedy użyć `docker run`, a kiedy Docker Compose.

## 12. Pytania kontrolne

Spróbuj odpowiedzieć przed przejrzeniem wyjaśnień:

1. Czy `docker build` uruchamia aplikację?
2. Jaka jest różnica między `RUN` i `CMD` w Dockerfile?
3. Czy `EXPOSE 8000` udostępnia port 8000 w przeglądarce hosta?
4. Co stanie się z named volume po usunięciu kontenera?
5. Do czego służy `127.0.0.1:8765:8000` przy `docker run -p`?
6. Dlaczego `localhost` w kontenerze nie oznacza hosta ani sąsiedniego kontenera?
7. Jaki problem rozwiązuje plik `.dockerignore`?
8. Dlaczego nie należy przechowywać sekretów w Dockerfile przez `ENV`?

### Odpowiedzi

1. Nie. Tworzy obraz; uruchomienie następuje przez `docker run` lub Compose.
2. `RUN` działa podczas **budowania obrazu**, a `CMD` określa domyślne polecenie przy **starcie kontenera**.
3. Nie. `EXPOSE` to metadane; do publikacji portu służy `-p` albo `ports` w Compose.
4. Zwykle pozostaje, o ile nie zostanie jawnie usunięty; nie jest częścią zapisywalnej warstwy kontenera.
5. Przekierowuje ruch z portu `8765` na lokalnym adresie hosta do portu `8000` w kontenerze.
6. `localhost` odnosi się do własnego stosu sieciowego danego kontenera; nazwy usług służą do komunikacji w sieci Dockera.
7. Ogranicza kontekst budowy i zapobiega dołączeniu zbędnych lub wrażliwych plików.
8. Mogą zostać zapisane w konfiguracji obrazu / historii i nie powinny być stosowane jako magazyn sekretów.

## 13. Ściąga Docker CLI

| Cel | Polecenie |
|---|---|
| Wersja klienta | `docker --version` |
| Informacje o Docker Engine | `docker info` |
| Pobierz obraz | `docker pull IMAGE` |
| Uruchom jednorazowo | `docker run --rm IMAGE` |
| Uruchom w tle | `docker run -d --name NAZWA IMAGE` |
| Opublikuj port | `docker run -p 127.0.0.1:8765:8000 IMAGE` |
| Zobacz kontenery | `docker ps` / `docker ps -a` |
| Zobacz obrazy | `docker image ls` |
| Logi | `docker logs NAZWA` |
| Proces w kontenerze | `docker exec NAZWA POLECENIE` |
| Zatrzymaj / usuń | `docker stop NAZWA` / `docker rm NAZWA` |
| Buduj obraz | `docker build -t NAZWA:TAG .` |
| Sprawdź wolumeny | `docker volume ls` |
| Sprawdź sieci | `docker network ls` |
| Compose: walidacja | `docker compose config` |
| Compose: start | `docker compose up -d --build` |
| Compose: status i logi | `docker compose ps` / `docker compose logs` |
| Compose: zatrzymaj projekt | `docker compose down` |

### Dokumentacja i dalsza nauka

- [Docker — oficjalny przewodnik startowy](https://docs.docker.com/get-started/)
- [Docker — pierwszy kontener](https://docs.docker.com/learn/content/run-your-first-container/)
- [Docker — własny Dockerfile](https://docs.docker.com/learn/content/write-your-first-dockerfile/)
- [Docker — publikowanie portów](https://docs.docker.com/get-started/docker-concepts/running-containers/publishing-ports/)
- [Docker Compose — oficjalne ćwiczenia](https://docs.docker.com/compose/gettingstarted/)